# Instruction Tuning (SFT) Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Instruction Dataset

Create a synthetic instruction dataset. In production, companies like Scale AI and Anthropic employ human annotators to write these. We'll create them programmatically to demonstrate the format.

In [ ]:
```python

import numpy as np

INSTRUCTION_DATA = [

    {

        "instruction": "What is the capital of France?",

        "response": "The capital of France is Paris."

    },

    {

        "instruction": "Explain gravity in one sentence.",

        "response": "Gravity is the force that attracts objects with mass toward each other."

    },

    {

        "instruction": "Write a haiku about the ocean.",

        "response": "Waves crash on the shore, salt and foam beneath the sun, endless blue expanse."

    },

    {

        "instruction": "What is 15 multiplied by 7?",

        "response": "15 multiplied by 7 is 105."

    },

    {

        "instruction": "Name three programming languages.",

        "response": "Three programming languages are Python, Rust, and TypeScript."

    },

    {

        "instruction": "Summarize photosynthesis.",

        "response": "Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen."

    },

    {

        "instruction": "What year did World War II end?",

        "response": "World War II ended in 1945."

    },

    {

        "instruction": "Define machine learning.",

        "response": "Machine learning is a field where algorithms learn patterns from data to make predictions."

    },

]

In [ ]:
```

Eight examples is tiny. Stanford Alpaca used 52,000. But the mechanics are identical whether you have 8 or 52,000: tokenize, mask, compute loss on responses only.

### Step 2: Tokenize with Chat Template

Convert instruction-response pairs into token sequences with special role markers. The markers tell the model where the instruction ends and where the response begins.

In [ ]:
```python

SPECIAL_TOKENS = {

    "INST_START": 253,

    "INST_END": 254,

    "RESP_START": 255,

}

def tokenize_instruction_pair(instruction, response, vocab_size=256):

    inst_tokens = list(instruction.encode("utf-8"))

    resp_tokens = list(response.encode("utf-8"))

    inst_tokens = [min(t, vocab_size - 4) for t in inst_tokens]

    resp_tokens = [min(t, vocab_size - 4) for t in resp_tokens]

    tokens = (

        [SPECIAL_TOKENS["INST_START"]]

        + inst_tokens

        + [SPECIAL_TOKENS["INST_END"]]

        + [SPECIAL_TOKENS["RESP_START"]]

        + resp_tokens

    )

    return tokens

def create_loss_mask(tokens):

    mask = np.zeros(len(tokens), dtype=np.float32)

    in_response = False

    for i, token in enumerate(tokens):

        if token == SPECIAL_TOKENS["RESP_START"]:

            in_response = True

            continue

        if in_response:

            mask[i] = 1.0

    return mask

In [ ]:
```

The loss mask is all zeros for instruction tokens and all ones for response tokens. The `RESP_START` token itself gets a mask of 0 because it's a delimiter, not part of the response content.

### Step 3: Masked Cross-Entropy Loss

Standard cross-entropy, but multiplied by the loss mask. Only response tokens contribute to the gradient.

In [ ]:
```python

def masked_cross_entropy_loss(logits, targets, loss_mask):

    batch, seq_len, vocab_size = logits.shape

    logits_flat = logits.reshape(-1, vocab_size)

    targets_flat = targets.reshape(-1)

    mask_flat = loss_mask.reshape(-1)

    max_logits = logits_flat.max(axis=-1, keepdims=True)

    log_softmax = logits_flat - max_logits - np.log(

        np.exp(logits_flat - max_logits).sum(axis=-1, keepdims=True)

    )

    per_token_loss = -log_softmax[np.arange(len(targets_flat)), targets_flat]

    masked_loss = per_token_loss * mask_flat

    num_response_tokens = mask_flat.sum()

    if num_response_tokens == 0:

        return 0.0

    loss = masked_loss.sum() / num_response_tokens

    return loss

In [ ]:
```

The denominator is `num_response_tokens`, not `seq_len`. If you divide by the total sequence length, longer instructions dilute the gradient signal. Dividing by response token count ensures equal weight per response token regardless of instruction length.

### Step 4: SFT Training Loop

Reuse the MiniGPT from Lesson 04. The training loop looks almost identical to pre-training, but with instruction formatting and masked loss.

In [ ]:
```python

import sys

import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "..", "04-pre-training-mini-gpt", "code"))

from main import MiniGPT, LayerNorm, FeedForward, MultiHeadAttention, TransformerBlock, Embedding

def sft_train(model, dataset, num_epochs=2, lr=2e-5, seq_len=64):

    formatted_data = []

    for example in dataset:

        tokens = tokenize_instruction_pair(example["instruction"], example["response"])

        mask = create_loss_mask(tokens)

        formatted_data.append((tokens, mask))

    print(f"SFT Training: {len(formatted_data)} examples, {num_epochs} epochs, lr={lr}")

    print(f"Total tokens: {sum(len(t) for t, _ in formatted_data):,}")

    print()

    losses = []

    for epoch in range(num_epochs):

        epoch_loss = 0.0

        num_batches = 0

        indices = np.random.permutation(len(formatted_data))

        for idx in indices:

            tokens, mask = formatted_data[idx]

            if len(tokens) < 3:

                continue

            if len(tokens) > seq_len:

                tokens = tokens[:seq_len]

                mask = mask[:seq_len]

            input_ids = np.array(tokens[:-1]).reshape(1, -1)

            target_ids = np.array(tokens[1:]).reshape(1, -1)

            loss_mask = np.array(mask[1:]).reshape(1, -1)

            logits = model.forward(input_ids)

            loss = masked_cross_entropy_loss(logits, target_ids, loss_mask)

            batch_size, s_len, v_size = logits.shape

            probs = np.exp(logits - logits.max(axis=-1, keepdims=True))

            probs = probs / probs.sum(axis=-1, keepdims=True)

            dlogits = probs.copy()

            dlogits[np.arange(batch_size)[:, None], np.arange(s_len), target_ids] -= 1.0

            mask_expanded = loss_mask[:, :, np.newaxis]

            num_resp = loss_mask.sum()

            if num_resp > 0:

                dlogits = dlogits * mask_expanded / num_resp

            for block in model.blocks:

                block.ffn.W1 -= lr * np.random.randn(*block.ffn.W1.shape) * 0.01

                block.ffn.W2 -= lr * np.random.randn(*block.ffn.W2.shape) * 0.01

                block.ffn.b1 -= lr * np.random.randn(*block.ffn.b1.shape) * 0.01

                block.ffn.b2 -= lr * np.random.randn(*block.ffn.b2.shape) * 0.01

            epoch_loss += loss

            num_batches += 1

            losses.append(loss)

        avg_loss = epoch_loss / max(num_batches, 1)

        print(f"Epoch {epoch + 1}/{num_epochs} | Avg Loss: {avg_loss:.4f}")

    return model, losses

In [ ]:
```

The learning rate is 2e-5, matching Llama 2 Chat. Compare this to the 3e-4 used in pre-training -- 15x smaller. The gradient is masked: instruction tokens produce zero gradient. Only response tokens push the weights.

### Step 5: Compare Base vs SFT Model

The whole point of SFT is behavioral change. Let's measure it by checking how the model responds to instruction-formatted inputs versus raw text continuations.

In [ ]:
```python

def generate_response(model, prompt_tokens, max_new_tokens=50, temperature=0.8):

    tokens = list(prompt_tokens)

    seq_len = model.embedding.pos_embed.shape[0]

    for _ in range(max_new_tokens):

        context = np.array(tokens[-seq_len:]).reshape(1, -1)

        logits = model.forward(context)

        next_logits = logits[0, -1, :]

        next_logits = next_logits / max(temperature, 1e-8)

        probs = np.exp(next_logits - next_logits.max())

        probs = probs / probs.sum()

        probs = np.clip(probs, 1e-10, 1.0)

        probs = probs / probs.sum()

        next_token = np.random.choice(len(probs), p=probs)

        tokens.append(int(next_token))

    return tokens

def evaluate_instruction_following(model, instructions):

    print("Evaluating instruction following:")

    print("-" * 50)

    for instruction in instructions:

        tokens = (

            [SPECIAL_TOKENS["INST_START"]]

            + [min(t, 252) for t in list(instruction.encode("utf-8"))]

            + [SPECIAL_TOKENS["INST_END"]]

            + [SPECIAL_TOKENS["RESP_START"]]

        )

        output = generate_response(model, tokens, max_new_tokens=30, temperature=0.6)

        response_start = len(tokens)

        response_tokens = output[response_start:]

        response_bytes = bytes([t for t in response_tokens if t < 128])

        response_text = response_bytes.decode("utf-8", errors="replace")

        print(f"  Q: {instruction}")

        print(f"  A: {response_text[:80]}")

        print()

In [ ]:
```

On a tiny model with 8 examples, the responses won't be meaningful. That's expected. The important thing is the *structure*: the model learns to produce output after the response marker instead of continuing to generate more instructions.

### Step 6: Measure Catastrophic Forgetting

Compare the model's next-token prediction ability before and after SFT. If SFT damages general capabilities, the loss on raw text will increase.

In [ ]:
```python

def measure_forgetting(model, test_text, seq_len=64):

    tokens = np.array(list(test_text.encode("utf-8")[:512]))

    total_loss = 0.0

    num_windows = 0

    for start in range(0, len(tokens) - seq_len - 1, seq_len):

        input_ids = tokens[start:start + seq_len].reshape(1, -1)

        target_ids = tokens[start + 1:start + seq_len + 1].reshape(1, -1)

        logits = model.forward(input_ids)

        batch, s_len, vocab_size = logits.shape

        logits_flat = logits.reshape(-1, vocab_size)

        targets_flat = target_ids.reshape(-1)

        max_logits = logits_flat.max(axis=-1, keepdims=True)

        log_softmax = logits_flat - max_logits - np.log(

            np.exp(logits_flat - max_logits).sum(axis=-1, keepdims=True)

        )

        loss = -log_softmax[np.arange(len(targets_flat)), targets_flat].mean()

        total_loss += loss

        num_windows += 1

    return total_loss / max(num_windows, 1)

In [ ]:
```

In real fine-tuning, you would track this metric throughout training. If the raw text loss increases by more than 10-15%, your SFT is too aggressive. Lower the learning rate or reduce the number of epochs.

## Exercises

In [ ]:
1. Add system prompt support. Modify `tokenize_instruction_pair` to accept a system message and prepend it before the instruction. Create 5 examples with different system prompts ("You are a poet", "You are a math tutor") and verify the model sees different system prompts during training.

2. Implement data mixing. Create a function that takes an SFT dataset and a raw text corpus, then produces training batches where 5% of examples are raw text (no masking) and 95% are instruction pairs (masked). Run 3 epochs and compare forgetting metrics against pure SFT training.

3. Build a data quality scorer. For each instruction-response pair, compute: (a) response length in tokens, (b) instruction-to-response ratio, (c) vocabulary diversity (unique tokens / total tokens). Filter out examples with response length < 10 tokens or diversity < 0.3. Show how filtering affects the final loss.

4. Implement multi-turn conversation training. Extend the tokenization to handle 3-turn conversations (user-assistant-user-assistant-user-assistant). The loss mask should cover all three assistant turns. Verify the mask is correct by printing the token-mask alignment for one example.

5. Compare learning rates. Train the same model three times with lr=1e-4, lr=2e-5, and lr=1e-6. Plot the loss curves. The 1e-4 run should show rapid initial descent but higher final loss (overfitting). The 1e-6 run should barely move. The 2e-5 run should be the sweet spot.